# 🎵 TuneWave Music Player — Milestone 1: Playlist
### Core Data Structures: Doubly Linked List (DLL) + Hash Map (Dictionary)

---

## 🎯 Milestone Goal & Problem Statement
In **TuneWave**, a music streaming app, users need seamless, jitter-free playback control. They move forward and back through tracks, add songs to their playlist, and remove songs even while music is playing.

### 📜 Formal Rules & Requirements:
1. **Uniqueness:** A song can appear **only once** in a playlist.
2. **Instant Lookup & Verification:** A hash map mapping `song_id -> Node` checks existence and finds songs in **$O(1)$** without linear scanning.
3. **Repeat Mode Semantics:**
   - With **repeat ON**: `next()` on the last song wraps back to the first song.
   - With **repeat OFF**: `next()` on the last song stays on the last song and returns `"end"`.
4. **Required Operations:**
   - `addSong(song)`
   - `removeSong(songId)`
   - `next()`
   - `previous()`
   - `toggleRepeat()`


## 🏗️ Architecture: Doubly Linked List + Hash Map

```
               +-------------------------------------------------+
               |              TuneWave Playlist                  |
               |  head: S1 | tail: S3 | current: S2 | repeat: off|
               +-------------------------------------------------+
                                       |
                   +-------------------+-------------------+
                   |                                       |
                   v                                       v
         [ Hash Map (song_map) ]                [ Doubly Linked List ]
       +---------+---------------+         +-----+     +-----+     +-----+
       | song_id |  Node Pointer |         | S1  |<--->| S2  |<--->| S3  |
       +---------+---------------+         +-----+     +-----+     +-----+
       | 'S1'    | ---> Node(S1) |            ^           ^           ^
       | 'S2'    | ---> Node(S2) |            |           |           |
       | 'S3'    | ---> Node(S3) |          [head]     [current]    [tail]
       +---------+---------------+
```

- **Hash Map** provides $O(1)$ membership checks and node lookups.
- **Doubly Linked List** provides $O(1)$ bidirectional sequential traversal and $O(1)$ node insertion/deletion without array shifting.


In [1]:
class Song:
    """Represents a track in TuneWave."""
    def __init__(self, song_id: str, title: str, artist: str, duration: int = 0):
        self.song_id = str(song_id)
        self.title = str(title)
        self.artist = str(artist)
        self.duration = duration  # duration in seconds

    def __repr__(self):
        return f"Song(id='{self.song_id}', title='{self.title}', artist='{self.artist}')"

    def __str__(self):
        return f"'{self.title}' by {self.artist} [{self.song_id}]"

    def __eq__(self, other):
        if isinstance(other, Song):
            return self.song_id == other.song_id
        elif isinstance(other, str):
            return self.song_id == other
        return False

    def __hash__(self):
        return hash(self.song_id)


class SongNode:
    """Doubly linked list node encapsulating a Song."""
    def __init__(self, song: Song):
        self.song = song
        self.prev = None
        self.next = None

    def __repr__(self):
        return f"SongNode({self.song.title})"


class Playlist:
    """
    Doubly Linked List + Hash Map implementation of TuneWave Playlist.
    Guarantees O(1) time complexity for all core operations.
    """
    def __init__(self, name: str = "My Playlist"):
        self.name = name
        self.head = None
        self.tail = None
        self.current = None
        self.repeat = False
        self.song_map = {}  # song_id -> SongNode
        self._size = 0

    def __len__(self):
        return self._size

    def addSong(self, song) -> bool:
        """
        Adds a song to the end of the playlist.
        Enforces uniqueness: a song can appear only once.
        Hash map checks duplicate in O(1) without scanning.
        """
        if not isinstance(song, Song):
            if isinstance(song, dict):
                song = Song(
                    song.get('song_id') or song.get('id'),
                    song.get('title', 'Unknown Title'),
                    song.get('artist', 'Unknown Artist'),
                    song.get('duration', 0)
                )
            elif isinstance(song, str):
                song = Song(song, song, 'Unknown Artist')
            else:
                raise TypeError(f"Unsupported song format: {type(song)}")

        # Rule 1: A song can appear only once in a playlist
        if song.song_id in self.song_map:
            return False  # Duplicate rejected in O(1)

        new_node = SongNode(song)

        if self.head is None:
            self.head = new_node
            self.tail = new_node
            self.current = new_node
        else:
            self.tail.next = new_node
            new_node.prev = self.tail
            self.tail = new_node

        self.song_map[song.song_id] = new_node
        self._size += 1
        return True

    def removeSong(self, songId: str) -> bool:
        """
        Removes a song by its songId in O(1) time.
        Uses hash map to locate node without scanning, then unlinks from DLL.
        """
        song_id_str = str(songId)
        node = self.song_map.get(song_id_str)
        if node is None:
            return False

        # If removing currently playing track, adjust current cursor
        if node == self.current:
            if node.next is not None:
                self.current = node.next
            elif node.prev is not None:
                self.current = node.prev
            else:
                self.current = None

        # Unlink head and tail pointers
        if node == self.head:
            self.head = node.next
            if self.head:
                self.head.prev = None

        if node == self.tail:
            self.tail = node.prev
            if self.tail:
                self.tail.next = None

        # Unlink internal node pointers
        if node.prev and node != self.head:
            node.prev.next = node.next
        if node.next and node != self.tail:
            node.next.prev = node.prev

        # Remove from hash map
        del self.song_map[song_id_str]
        self._size -= 1

        # Clear node references
        node.prev = None
        node.next = None
        return True

    def next(self):
        """
        Advances to the next track.
        - With repeat ON: next() on last song wraps to the first.
        - With repeat OFF: stays on last song and returns 'end'.
        """
        if self.current is None:
            return 'end'

        if self.current.next is not None:
            self.current = self.current.next
            return self.current.song

        # At the last song
        if self.repeat:
            self.current = self.head
            return self.current.song if self.current else 'end'
        else:
            return 'end'

    def previous(self):
        """
        Navigates to the previous track.
        """
        if self.current is None:
            return None

        if self.current.prev is not None:
            self.current = self.current.prev
            return self.current.song

        # At the first song
        if self.repeat:
            self.current = self.tail
            return self.current.song if self.current else None
        else:
            return self.current.song

    def toggleRepeat(self) -> bool:
        """Toggles repeat mode and returns current state."""
        self.repeat = not self.repeat
        return self.repeat

    def getCurrentSong(self):
        """Returns the currently active song without moving the cursor."""
        return self.current.song if self.current else None

    def display(self):
        """Displays playlist contents with cursor indicator."""
        curr = self.head
        songs = []
        while curr:
            marker = '▶ [PLAYING]' if curr == self.current else '            '
            songs.append(f"{marker} {curr.song}")
            curr = curr.next
        repeat_status = 'ON' if self.repeat else 'OFF'
        return f"Playlist '{self.name}' (Repeat: {repeat_status}, Total Songs: {len(self)}):\n" + '\n'.join(songs)


### 🧪 Interactive Demo 1: Creating a Playlist and Verifying Duplicate Prevention
Let's populate our TuneWave playlist and attempt to insert duplicates.


In [2]:
playlist = Playlist("Chill Vibes")

# Create songs
s1 = Song("S1", "Midnight City", "M83", 244)
s2 = Song("S2", "Starboy", "The Weeknd", 230)
s3 = Song("S3", "Levitating", "Dua Lipa", 203)

print("Adding S1:", playlist.addSong(s1))
print("Adding S2:", playlist.addSong(s2))
print("Adding S3:", playlist.addSong(s3))

# Attempt duplicate addition
duplicate_result = playlist.addSong(s1)
print("\nAttempting duplicate addition of S1:", duplicate_result)
print("Playlist size after duplicate attempt:", len(playlist))

print("\nCurrent State:")
print(playlist.display())


Adding S1: True
Adding S2: True
Adding S3: True

Attempting duplicate addition of S1: False
Playlist size after duplicate attempt: 3

Current State:
Playlist 'Chill Vibes' (Repeat: OFF, Total Songs: 3):
▶ [PLAYING] 'Midnight City' by M83 [S1]
             'Starboy' by The Weeknd [S2]
             'Levitating' by Dua Lipa [S3]


### 🔁 Interactive Demo 2: Navigation & Repeat Mode
Demonstrating `next()`, `previous()`, and the difference between `repeat=False` and `repeat=True`.


In [3]:
print("Currently Playing:", playlist.getCurrentSong())

print("\n-- Advancing through playlist with repeat OFF --")
print("1st next():", playlist.next())
print("2nd next():", playlist.next())
print("3rd next() (at terminal track):", playlist.next())
print("Cursor retained at terminal track:", playlist.getCurrentSong())

print("\n-- Navigating Back --")
print("previous():", playlist.previous())
print("previous():", playlist.previous())
print("previous() (at head track):", playlist.previous())

print("\n-- Enabling Repeat Mode --")
new_repeat_state = playlist.toggleRepeat()
print("Repeat mode enabled?:", new_repeat_state)

# Fast-forward to last song
playlist.next()
playlist.next()
print("\nAt last song:", playlist.getCurrentSong())
print("next() with repeat ON (should wrap to head):", playlist.next())
print("Currently playing:", playlist.getCurrentSong())


Currently Playing: 'Midnight City' by M83 [S1]

-- Advancing through playlist with repeat OFF --
1st next(): 'Starboy' by The Weeknd [S2]
2nd next(): 'Levitating' by Dua Lipa [S3]
3rd next() (at terminal track): end
Cursor retained at terminal track: 'Levitating' by Dua Lipa [S3]

-- Navigating Back --
previous(): 'Starboy' by The Weeknd [S2]
previous(): 'Midnight City' by M83 [S1]
previous() (at head track): 'Midnight City' by M83 [S1]

-- Enabling Repeat Mode --
Repeat mode enabled?: True

At last song: 'Levitating' by Dua Lipa [S3]
next() with repeat ON (should wrap to head): 'Midnight City' by M83 [S1]
Currently playing: 'Midnight City' by M83 [S1]


### 🗑️ Interactive Demo 3: Removing Songs Dynamically in O(1)
Testing node removal at head, middle, tail, and while a song is currently playing.


In [4]:
print("Initial playlist:")
print(playlist.display())

# Current song is S1. Let's remove S1 while it's playing!
print("\nRemoving active song S1:", playlist.removeSong("S1"))
print("Current track automatically shifted to:", playlist.getCurrentSong())

# Remove non-existent song
print("\nRemoving non-existent S999:", playlist.removeSong("S999"))

print("\nState after removal:")
print(playlist.display())


Initial playlist:
Playlist 'Chill Vibes' (Repeat: ON, Total Songs: 3):
▶ [PLAYING] 'Midnight City' by M83 [S1]
             'Starboy' by The Weeknd [S2]
             'Levitating' by Dua Lipa [S3]

Removing active song S1: True
Current track automatically shifted to: 'Starboy' by The Weeknd [S2]

Removing non-existent S999: False

State after removal:
Playlist 'Chill Vibes' (Repeat: ON, Total Songs: 2):
▶ [PLAYING] 'Starboy' by The Weeknd [S2]
             'Levitating' by Dua Lipa [S3]


### 🛡️ Automated Comprehensive Test Suite
Validating all rules, constraints, edge cases, and invariants.


In [5]:
def run_all_tests():
    p = Playlist("Test Suite")
    t1 = Song("T1", "Track 1", "Artist 1")
    t2 = Song("T2", "Track 2", "Artist 2")
    t3 = Song("T3", "Track 3", "Artist 3")

    # 1. Test addSong and length
    assert p.addSong(t1) is True
    assert p.addSong(t2) is True
    assert p.addSong(t3) is True
    assert len(p) == 3
    assert p.getCurrentSong() == t1

    # 2. Test duplicate rejection (O(1))
    assert p.addSong(t1) is False
    assert p.addSong(Song("T2", "Duplicate Title", "Other Artist")) is False
    assert len(p) == 3

    # 3. Test next with repeat OFF
    assert p.next() == t2
    assert p.next() == t3
    assert p.next() == 'end'  # Rule: stays on last song and returns 'end'
    assert p.getCurrentSong() == t3

    # 4. Test previous
    assert p.previous() == t2
    assert p.previous() == t1
    assert p.previous() == t1  # Stays at head

    # 5. Test toggleRepeat and next() wrapping
    assert p.toggleRepeat() is True
    assert p.next() == t2
    assert p.next() == t3
    assert p.next() == t1  # Rule: wraps to first song
    assert p.getCurrentSong() == t1

    # 6. Test removeSong (middle)
    assert p.removeSong("T2") is True
    assert len(p) == 2
    assert "T2" not in p.song_map

    # 7. Test remove current song
    assert p.getCurrentSong() == t1
    assert p.removeSong("T1") is True
    assert p.getCurrentSong() == t3

    # 8. Test remove until empty
    assert p.removeSong("T3") is True
    assert len(p) == 0
    assert p.head is None and p.tail is None and p.current is None
    assert p.next() == 'end'
    assert p.previous() is None

    # 9. Test single song playlist
    p2 = Playlist("Single Song")
    assert p2.addSong(t1) is True
    assert p2.getCurrentSong() == t1
    assert p2.next() == 'end'
    p2.toggleRepeat()
    assert p2.next() == t1  # wraps to itself

    print("🎉 ALL TESTS PASSED SUCCESSFULLY! Milestone 1 rules 100% verified.")

run_all_tests()


🎉 ALL TESTS PASSED SUCCESSFULLY! Milestone 1 rules 100% verified.


## 📊 Complexity & Performance Summary

| Operation | Time Complexity | Space Complexity | Verification |
| :--- | :---: | :---: | :--- |
| `addSong(song)` | **$O(1)$** | **$O(1)$** | Hash lookup + DLL tail append |
| `removeSong(songId)` | **$O(1)$** | **$O(1)$** | Hash map lookup + pointer unlinking |
| `next()` | **$O(1)$** | **$O(1)$** | Direct pointer advance / wrap |
| `previous()` | **$O(1)$** | **$O(1)$** | Direct pointer rewind |
| `toggleRepeat()` | **$O(1)$** | **$O(1)$** | Atomic boolean negation |

Milestone 1 is complete and production-ready!
